In [ ]:
# -*- coding: utf-8 -*-
"""scGPT_finetune_real_data.ipynb

# scGPT Pretrained Fine-Tuning Benchmark — Phase 8

**What this notebook tests:**
Fine-tuning the *pretrained* scGPT transformer (12-layer, 512-dim, Genecorpus-30M equivalent)
on real datasets (80/20 train/test split). Instead of simulated scenarios with replicates,
this notebook iterates over all independent `.h5ad` samples in the `h5ad_real` directory.

**GPU:** L4 (22.5 GB VRAM).
**Crash safety:** Each run saves immediately to Drive. Re-running skips completed runs.
"""

# @title Mount Google Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q scanpy anndata scikit-learn

In [ ]:
# @title Imports & Environment Config
import os
import glob

# ── Allocator config — reduces fragmentation, set before any CUDA ops ─────
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import sys, time, warnings, random, gc, json, math, urllib.request
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.sparse import issparse

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import train_test_split
from sklearn.metrics import (cohen_kappa_score, f1_score,
                              matthews_corrcoef, accuracy_score)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {props.name}")
    print(f"VRAM   : {props.total_memory / 1e9:.1f} GB")

In [ ]:
# @title Configuration & Hyperparameters

# ── EDIT THESE TO MATCH YOUR DRIVE ────────────────────────────────────────
DRIVE_ROOT      = "/content/drive/MyDrive/Thesis FM Computing"
H5AD_DIR        = f"{DRIVE_ROOT}/h5ad_real"
RESULTS_DIR     = f"{DRIVE_ROOT}/phase8_results"
CKPT_DIR        = f"{DRIVE_ROOT}/phase8_checkpoints/scgpt_finetune"

# Official scGPT Pretrained Weights
PRETRAINED_CKPT = f"{DRIVE_ROOT}/scGPT_human_best_model.pt"
# ─────────────────────────────────────────────────────────────────────────

MODEL_NAME = "scGPT_pretrained"
TRAIN_SEED = 42

# scGPT Fine-tuning Architecture Defaults
MAX_SEQ_LEN  = 1200
N_BINS       = 51

# Fine-tuning hyperparameters
BATCH_SIZE   = 2
ACCUM_STEPS  = 4
MAX_EPOCHS   = 30
LR           = 1e-4
PATIENCE     = 5
WEIGHT_DECAY = 1e-4

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,    exist_ok=True)

# Dynamically find all samples in the directory
sample_paths = sorted(glob.glob(os.path.join(H5AD_DIR, "*.h5ad")))
sample_paths =[p for p in sample_paths if "_scgpt" not in p]
SAMPLES =[os.path.basename(p).replace(".h5ad", "") for p in sample_paths]

print(f"Found {len(SAMPLES)} samples in {H5AD_DIR}")
print(f"RESULTS_DIR : {RESULTS_DIR}")
print(f"CKPT_DIR    : {CKPT_DIR}")

In [ ]:
# @title scGPT Vocabulary & Pretrained Weights Fetching
import os
import json
import urllib.request

VOCAB_PATH = "/tmp/scgpt_vocab.json"

def get_scgpt_vocab():
    VOCAB_URLS =[
        "https://huggingface.co/wanglab/scGPT-human/resolve/main/vocab.json",
        "https://huggingface.co/perturblab/scgpt-human/resolve/main/vocab.json"
    ]

    if not os.path.exists(VOCAB_PATH):
        print("Downloading official scGPT vocabulary...")
        success = False
        for url in VOCAB_URLS:
            try:
                req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req) as response, open(VOCAB_PATH, 'wb') as out_file:
                    out_file.write(response.read())
                print(f"  Successfully downloaded vocab from Hugging Face.")
                success = True
                break
            except Exception as e:
                print(f"  [WARN] Failed to download from {url}: {e}")

        if not success:
            raise RuntimeError("Failed to download scGPT vocab.json.")

    with open(VOCAB_PATH, "r") as f:
        vocab = json.load(f)

    genes = sorted([g for g in vocab.keys() if not g.startswith("<")])
    print(f"Loaded scGPT vocabulary: {len(vocab)} total tokens | {len(genes)} real genes")
    return vocab, genes

def download_scgpt_pretrained():
    MODEL_URL = "https://huggingface.co/wanglab/scGPT-human/resolve/main/best_model.pt"

    if not os.path.exists(PRETRAINED_CKPT):
        print(f"Pretrained weights not found at {PRETRAINED_CKPT}.")
        print("Downloading official scGPT-human weights from Hugging Face (~340 MB)...")
        try:
            req = urllib.request.Request(MODEL_URL, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as response, open(PRETRAINED_CKPT, 'wb') as out_file:
                out_file.write(response.read())
            print("  Successfully downloaded pretrained weights.")
        except Exception as e:
            raise RuntimeError(f"Failed to download pretrained weights: {e}")
    else:
        print("Pretrained weights found locally.")

SCGPT_VOCAB, GENE_PANEL = get_scgpt_vocab()
download_scgpt_pretrained()

In [ ]:
# @title Data Loading & Tokenisation Helpers

def ensure_raw_counts(adata):
    # Raw integer counts are stored in adata.layers['counts']; adata.X is z-scored (has negatives).
    # Prefer the counts layer, then .raw, then fall back to X (guarded).
    if "counts" in adata.layers:
        X, var = adata.layers["counts"], adata.var
    elif adata.raw is not None:
        X, var = adata.raw.X, adata.raw.var
    else:
        X, var = adata.X, adata.var
    if issparse(X):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float32)
    if X.min() < 0:
        raise ValueError(
            "Count matrix has negative values \u2014 layers['counts'] is not raw counts "
            "(adata.X is z-scored). Cannot tokenize."
        )
    return ad.AnnData(
        X   = np.round(X).astype(np.float32),
        obs = adata.obs.copy(),
        var = var.copy(),
        uns = adata.uns.copy()
    )

def load_sample(sample_name):
    """Load real data sample."""
    raw_path = os.path.join(H5AD_DIR, f"{sample_name}.h5ad")

    if not os.path.exists(raw_path):
        raise FileNotFoundError(f"Cannot find: {raw_path}")

    adata = sc.read_h5ad(raw_path)
    adata = ensure_raw_counts(adata)

    # Fix: Rename 'Ground_Truth_Celltype' to 'cell_type' if present
    if "Ground_Truth_Celltype" in adata.obs.columns and "cell_type" not in adata.obs.columns:
        adata.obs["cell_type"] = adata.obs["Ground_Truth_Celltype"]
        print("  [FIX] Renamed 'Ground_Truth_Celltype' to 'cell_type'.")

    assert "cell_type" in adata.obs.columns, (
        f"'cell_type' column missing. Available: {list(adata.obs.columns)}"
    )

    n_types = adata.obs["cell_type"].nunique()
    print(f"  Loaded {sample_name}: {adata.shape[0]} cells × {adata.shape[1]} genes | {n_types} cell types")
    return adata

def stratified_split(adata, test_size=0.2):
    # Fix: Remove cells with NaN cell_type
    initial_count = len(adata)
    adata = adata[~adata.obs["cell_type"].isna()].copy()
    if len(adata) < initial_count:
        print(f"  [FIX] Removed {initial_count - len(adata)} cells with missing cell types.")

    counts = adata.obs["cell_type"].value_counts()
    singletons = counts[counts < 2].index.tolist()
    if singletons:
        print(f"  [FIX] Removing singleton classes: {singletons}")
        adata = adata[~adata.obs["cell_type"].isin(singletons)].copy()

    labels = adata.obs["cell_type"].values
    idx    = np.arange(len(labels))
    tr_idx, te_idx = train_test_split(idx, test_size=test_size, stratify=labels, random_state=TRAIN_SEED)
    return adata[tr_idx].copy(), adata[te_idx].copy()

def encode_labels(train_a, test_a):
    all_types = sorted(train_a.obs["cell_type"].unique())
    label2id  = {t: i for i, t in enumerate(all_types)}
    id2label  = {i: t for t, i in label2id.items()}
    y_train   = np.array([label2id[t] for t in train_a.obs["cell_type"]])
    y_test    = np.array([label2id[t] for t in test_a.obs["cell_type"]])
    return label2id, id2label, y_train, y_test

def to_dense(adata):
    X = adata.X
    if issparse(X): X = X.toarray()
    return X.astype(np.float32)

def tokenise_for_scgpt(X_raw, gene_names, vocab, max_len=MAX_SEQ_LEN, n_bins=N_BINS):
    n_cells, n_genes = X_raw.shape

    # Metrics on matching
    matched = sum(1 for g in gene_names if g in vocab)
    print(f"  Matched to scGPT vocab: {matched}/{len(gene_names)} genes ({(matched/len(gene_names))*100:.1f}%)")

    lib = X_raw.sum(axis=1, keepdims=True)
    lib = np.where(lib == 0, 1.0, lib)
    X_norm = np.log1p((X_raw / lib) * 10_000.0)

    non_zeros = X_norm[X_norm > 0]
    if len(non_zeros) == 0:
        bins = np.linspace(0, 1, n_bins)
    else:
        bins = np.linspace(non_zeros.min(), non_zeros.max(), n_bins)

    X_binned = np.digitize(X_norm, bins)
    X_binned[X_norm == 0] = 0

    # Fix: Provide default values (0) if special tokens are missing from dict
    unk_id = vocab.get("<unk>", 0)
    gene_ids = np.array([vocab.get(g, unk_id) for g in gene_names], dtype=np.int64)

    pad_id = vocab.get("<pad>", 0)
    cls_id = vocab.get("<cls>", 0)

    out_ids  = np.full((n_cells, max_len), pad_id, dtype=np.int64)
    out_vals = np.zeros((n_cells, max_len), dtype=np.int64)
    out_mask = np.ones((n_cells, max_len), dtype=bool)

    for i in range(n_cells):
        row_binned = X_binned[i]
        nz_idx = np.nonzero(row_binned)[0]
        sorted_nz = nz_idx[np.argsort(row_binned[nz_idx])[::-1]]
        sorted_nz = sorted_nz[:max_len - 1]
        seq_len = len(sorted_nz) + 1

        out_ids[i, 0] = cls_id
        out_vals[i, 0] = 0
        out_mask[i, 0] = False

        out_ids[i, 1:seq_len] = gene_ids[sorted_nz]
        out_vals[i, 1:seq_len] = row_binned[sorted_nz]
        out_mask[i, 1:seq_len] = False

    return torch.tensor(out_ids, dtype=torch.long), torch.tensor(out_vals, dtype=torch.long), torch.tensor(out_mask, dtype=torch.bool)

In [ ]:
# @title scGPT Pretrained Architecture Definitions

class ContinuousValueEncoder(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.2):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.linear1 = nn.Linear(1, d_model)
        self.activation = nn.ReLU()
        self.linear2 = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(-1).float()
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        x = self.norm(x)
        return self.dropout(x)

class GeneEncoder(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int, padding_idx: int):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, embedding_dim, padding_idx=padding_idx)
        self.enc_norm = nn.LayerNorm(embedding_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.embedding(x)
        x = self.enc_norm(x)
        return x

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.2, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:x.size(1)].transpose(0, 1)
        return self.dropout(x)

class ClsDecoder(nn.Module):
    def __init__(self, d_model: int, n_cls: int, nlayers: int = 3, activation: callable = nn.ReLU):
        super().__init__()
        self._decoder = nn.ModuleList()
        for i in range(nlayers - 1):
            self._decoder.append(nn.Linear(d_model, d_model))
            self._decoder.append(activation())
            self._decoder.append(nn.LayerNorm(d_model))
        self.out_layer = nn.Linear(d_model, n_cls)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for mod in self._decoder:
            x = mod(x)
        return self.out_layer(x)

class scGPTPretrained(nn.Module):
    def __init__(self, vocab_size, num_classes, pad_token_id, pretrained_path=None):
        super().__init__()
        self.d_model = 512

        # 1. Parallel Encoders
        # We deliberately name this 'encoder' instead of 'gene_encoder' so it perfectly
        # matches the state_dict keys inside scGPT's official best_model.pt
        self.encoder       = GeneEncoder(vocab_size, self.d_model, padding_idx=pad_token_id)
        self.value_encoder = ContinuousValueEncoder(self.d_model, dropout=0.2)
        self.pos_encoder   = PositionalEncoding(self.d_model, dropout=0.2, max_len=MAX_SEQ_LEN)

        # 2. 12-layer Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_model,
            nhead=8,
            dim_feedforward=512,
            dropout=0.2,
            batch_first=True,
            norm_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=12)

        # 3. Classification Head (Randomly Initialised for downstream task)
        self.cls_decoder = ClsDecoder(self.d_model, num_classes, nlayers=3)

        self._init_weights()

        if pretrained_path and os.path.exists(pretrained_path):
            self._load_pretrained(pretrained_path)

        n_params = sum(p.numel() for p in self.parameters())
        print(f"  scGPT Architecture: {num_classes} classes, {n_params:,} params — Pretrained init")

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _load_pretrained(self, pretrained_path):
        print(f"  Loading pretrained weights from {pretrained_path}...")
        ckpt = torch.load(pretrained_path, map_location="cpu")

        # scGPT dicts sometimes save under 'model_state_dict'
        if 'model_state_dict' in ckpt:
            ckpt = ckpt['model_state_dict']

        # Remove prefix if present (like module.)
        new_dict = {}
        for k, v in ckpt.items():
            new_k = k.replace("module.", "")
            new_dict[new_k] = v

        # strict=False safely ignores missing keys (like our newly initialized cls_decoder)
        missing, unexpected = self.load_state_dict(new_dict, strict=False)
        matched = len(new_dict) - len(unexpected)
        print(f"  Pretrained backbone: {matched}/{len(new_dict)} checkpoint tensors matched "
              f"| {len(missing)} missing (classifier head expected) | {len(unexpected)} unexpected.")
        if matched == 0:
            print("  [WARNING] 0 weights matched -> model is effectively RANDOM INIT. Check key names!")

    def forward(self, input_ids, values, src_key_padding_mask):
        gene_emb = self.encoder(input_ids)
        val_emb  = self.value_encoder(values)

        src = gene_emb + val_emb
        src = self.pos_encoder(src)

        output = self.transformer_encoder(src, src_key_padding_mask=src_key_padding_mask)

        # Extract <cls> token at sequence index 0
        cls_rep = output[:, 0, :]
        logits = self.cls_decoder(cls_rep)

        return logits

In [ ]:
# @title Training, Evaluation, and Save Utilities

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total, n = 0.0, 0
    optimizer.zero_grad()

    for step, (ids, vals, mask, yb) in enumerate(loader):
        ids, vals, mask, yb = ids.to(DEVICE), vals.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE)

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = model(ids, vals, src_key_padding_mask=mask)
            loss = criterion(logits.float(), yb) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total += loss.item() * ACCUM_STEPS * len(yb)
        n += len(yb)

    return total / n

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total, n, preds = 0.0, 0,[]
    for ids, vals, mask, yb in loader:
        ids, vals, mask, yb = ids.to(DEVICE), vals.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE)

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = model(ids, vals, src_key_padding_mask=mask)

        total += criterion(logits.float(), yb).item() * len(yb); n += len(yb)
        preds.extend(logits.argmax(1).cpu().numpy())
    return total / n, np.array(preds)

def fit_model(model, X_tr_ids, X_tr_vals, X_tr_mask, y_tr, X_te_ids, X_te_vals, X_te_mask, y_te, sample_name):
    tr_dl = DataLoader(TensorDataset(X_tr_ids, X_tr_vals, X_tr_mask, torch.tensor(y_tr, dtype=torch.long)),
                       batch_size=BATCH_SIZE, shuffle=True, pin_memory=(DEVICE.type == "cuda"))
    te_dl = DataLoader(TensorDataset(X_te_ids, X_te_vals, X_te_mask, torch.tensor(y_te, dtype=torch.long)),
                       batch_size=BATCH_SIZE, shuffle=False, pin_memory=(DEVICE.type == "cuda"))

    w = 1.0 / (np.bincount(y_tr) + 1e-6)
    w = w / w.sum() * len(w)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(DEVICE))

    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    ckpt = os.path.join(CKPT_DIR, f"scgpt_finetune_{sample_name}_best.pt")

    best_val, pat = float("inf"), 0
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss = train_epoch(model, tr_dl, optimizer, criterion)
        vl_loss, _ = eval_epoch(model, te_dl, criterion)
        scheduler.step()

        if vl_loss < best_val:
            best_val, pat = vl_loss, 0
            torch.save(model.state_dict(), ckpt)
        else:
            pat += 1

        if epoch % 5 == 0:
            print(f"    Epoch {epoch:02d} | train={tr_loss:.4f} val={vl_loss:.4f} | patience={pat}/{PATIENCE}")

        if pat >= PATIENCE:
            print(f"    Early stop at epoch {epoch}")
            break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    _, final_preds = eval_epoch(model, te_dl, criterion)

    runtime = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2 if DEVICE.type == "cuda" else 0
    print(f"    Done in {runtime/60:.1f} min | peak GPU: {peak_mb:.0f} MB")
    return final_preds, runtime, peak_mb

In [ ]:
def compute_metrics(y_true, y_pred, id2label):
    kappa    = cohen_kappa_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mcc      = matthews_corrcoef(y_true, y_pred)
    acc      = accuracy_score(y_true, y_pred)

    unique, counts = np.unique(y_true, return_counts=True)
    rarest = unique[np.argmin(counts)]
    rare_f1 = f1_score((y_true == rarest).astype(int),
                       (y_pred == rarest).astype(int), zero_division=0)
    return {
        "kappa":          round(float(kappa),    4),
        "macro_f1":       round(float(macro_f1), 4),
        "mcc":            round(float(mcc),      4),
        "accuracy":       round(float(acc),      4),
        "rare_type_f1":   round(float(rare_f1),  4),
        "rare_type_name": id2label[rarest],
        "rare_type_n":    int(counts.min()),
        "n_test":         int(len(y_true)),
        "n_classes":      int(len(id2label)),
    }

def print_metrics(m):
    print(f"  κ (kappa)    : {m['kappa']:.4f}")
    print(f"  Macro F1     : {m['macro_f1']:.4f}")
    print(f"  MCC          : {m['mcc']:.4f}")
    print(f"  Accuracy     : {m['accuracy']:.4f}")
    print(f"  Rare type F1 : {m['rare_type_f1']:.4f}  ({m['rare_type_name']}, n_test={m['rare_type_n']})")

In [ ]:
CSV_PATH = os.path.join(RESULTS_DIR, "phase8_results_scGPT.csv")

def save_result(sample_name, metrics, predictions, runtime_s, peak_gpu_mb):
    row = {
        "model":       MODEL_NAME,
        "sample":      sample_name,
        "condition":   "pretrained_finetune",
        "runtime_s":   round(runtime_s, 1),
        "peak_gpu_mb": round(peak_gpu_mb, 0),
        **metrics
    }
    row_df = pd.DataFrame([row])
    if os.path.exists(CSV_PATH):
        row_df.to_csv(CSV_PATH, mode="a", header=False, index=False)
    else:
        row_df.to_csv(CSV_PATH, mode="w", header=True,  index=False)

    pred_fname = f"{MODEL_NAME}_{sample_name}_preds.npy"
    np.save(os.path.join(RESULTS_DIR, pred_fname), predictions)
    print(f"  Saved metrics → phase8_results_scGPT.csv")
    print(f"  Saved preds   → {pred_fname}")

def check_done(sample_name):
    if not os.path.exists(CSV_PATH): return False
    df = pd.read_csv(CSV_PATH)
    return ((df["model"]  == MODEL_NAME)  &
            (df["sample"] == sample_name) &
            (df["condition"] == "pretrained_finetune")).any()

In [ ]:
# @title Main Execution Loop

completed, total = 0, len(SAMPLES)

print("=" * 65)
print(f"scGPT Pretrained Fine-Tuning | {total} real samples")
print("=" * 65)

for sample in SAMPLES:
    print(f"\n{'─'*65}")
    print(f"[{completed+1}/{total}]  Sample: {sample}")
    print(f"{'─'*65}")

    if DEVICE.type == "cuda":
        free_gb, total_gb =[x/1e9 for x in torch.cuda.mem_get_info()]
        print(f"  GPU memory: {free_gb:.1f} GB free / {total_gb:.1f} GB total")

    if check_done(sample):
        print(f"  [SKIP] Already in results CSV")
        completed += 1
        continue

    try:
        adata = load_sample(sample)
    except FileNotFoundError as e:
        print(f"  [WARN] {e}\n  Skipping sample.")
        continue

    train_a, test_a = stratified_split(adata)
    label2id, id2label, y_train, y_test = encode_labels(train_a, test_a)
    n_classes = len(label2id)
    print(f"  Train: {len(y_train)} cells | Test: {len(y_test)} cells | Classes: {n_classes}")

    print("  Tokenising...")
    gene_names = list(adata.var_names)

    X_train_dense = to_dense(train_a)
    del train_a; gc.collect()
    X_tr_ids, X_tr_vals, X_tr_mask = tokenise_for_scgpt(X_train_dense, gene_names, SCGPT_VOCAB)
    del X_train_dense; gc.collect()

    X_test_dense = to_dense(test_a)
    del adata, test_a; gc.collect()
    X_te_ids, X_te_vals, X_te_mask = tokenise_for_scgpt(X_test_dense, gene_names, SCGPT_VOCAB)
    del X_test_dense; gc.collect()

    VOCAB_SIZE = len(SCGPT_VOCAB)
    PAD_TOKEN_ID = SCGPT_VOCAB.get("<pad>", 0)

    model = scGPTPretrained(
        vocab_size=VOCAB_SIZE,
        num_classes=n_classes,
        pad_token_id=PAD_TOKEN_ID,
        pretrained_path=PRETRAINED_CKPT
    ).to(DEVICE)

    print("  Training...")
    preds, runtime, peak_mb = fit_model(
        model, X_tr_ids, X_tr_vals, X_tr_mask, y_train,
        X_te_ids, X_te_vals, X_te_mask, y_test, sample
    )

    metrics = compute_metrics(y_test, preds, id2label)
    print("  Results:")
    print_metrics(metrics)

    save_result(sample, metrics, preds, runtime, peak_mb)
    completed += 1

    del model, X_tr_ids, X_tr_vals, X_tr_mask, X_te_ids, X_te_vals, X_te_mask
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    gc.collect()

print("\n" + "=" * 65)
print(f"Complete: {completed}/{total} runs")
print("=" * 65)


In [ ]:
# @title Print Results Summary

if os.path.exists(CSV_PATH):
    df   = pd.read_csv(CSV_PATH)
    mine = df[df["model"] == MODEL_NAME].copy()

    if len(mine) > 0:
        print("=== Per-run results ===")
        cols =["sample", "kappa", "macro_f1", "rare_type_f1", "accuracy", "runtime_s", "peak_gpu_mb"]
        print(mine[cols].sort_values("sample").to_string(index=False))

        print(f"\nTotal samples recorded: {len(mine)}/{total}")
        print(f"Total runtime         : {mine['runtime_s'].sum()/3600:.2f} hours")
        print(f"Mean κ (all samples)  : {mine['kappa'].mean():.4f}")
    else:
        print("No results yet — run the main loop first.")
else:
    print(f"No CSV found at {CSV_PATH}")

In [ ]:
from google.colab import runtime
runtime.unassign()